# Stage 5 — calendar features

This notebook is the Stage 5 demo surface. It runs the **same** `LinearModel` class twice — once on the Stage 3 five-column `weather_only` feature table, once on the 49-column `weather_calendar` extension — and shows what domain knowledge bought us.

### Why this matters

The Stage 4 linear baseline lost cleanly to NESO's day-ahead forecast because weather alone cannot explain GB demand: consumption has a strong weekly rhythm (weekday vs weekend), a daily shape (breakfast / evening peaks), an annual seasonal envelope, and conspicuous holes on public holidays. Stage 5 encodes those four effects as one-hot dummies and appends them to the feature table — without touching `LinearModel`, `harness.evaluate`, `compare_on_holdout`, `SplitterConfig`, or any of the Stage 4 plumbing. The only difference between the two runs is which feature set the Hydra config resolves to.

### What this notebook does

1. Loads the two feature tables via `assembler.load` and `assembler.load_calendar`.
2. Fits `LinearModel` on `weather_only` and runs it through `harness.evaluate`.
3. Fits `LinearModel` on `weather_calendar` and prints `results.summary()` — **this is the teaching-moment table**; facilitators point at the hour-of-day dummies to read the load profile off the coefficients.
4. Runs both feature sets through the three-way `compare_on_holdout` if the NESO day-ahead forecast archive is cached locally.
5. Plots residuals for both models on a shared test week so the weekly ripple the weather-only model misses is visibly flattened by calendar features.

Acceptance criterion 12: under 120 seconds with warm caches on a laptop (plan D8). The splitter uses `step=168` (weekly stride) and `min_train_periods=720` (30 days) to keep the notebook inside that budget; the CLI (`python -m bristol_ml.train features=weather_calendar`) runs the full daily-stride configuration.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # cache_dir values resolve against cwd

import matplotlib.dates as mdates  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from bristol_ml import load_config  # noqa: E402
from bristol_ml.evaluation.benchmarks import compare_on_holdout  # noqa: E402
from bristol_ml.evaluation.harness import evaluate  # noqa: E402
from bristol_ml.evaluation.metrics import METRIC_REGISTRY  # noqa: E402
from bristol_ml.features import assembler  # noqa: E402
from bristol_ml.features.calendar import CALENDAR_VARIABLE_COLUMNS  # noqa: E402
from bristol_ml.ingestion import neso_forecast as neso_forecast_mod  # noqa: E402
from bristol_ml.models.linear import LinearModel  # noqa: E402

from conf._schemas import LinearConfig  # noqa: E402

# Plan D8 — narrow the first fold's training window and stride weekly so the
# notebook finishes in under 120 s.  The CLI path inherits the defaults
# (one-year train, daily stride) for reporting-quality output.
cfg_wonly = load_config(
    config_path=REPO_ROOT / "conf",
    overrides=[
        "evaluation.rolling_origin.min_train_periods=720",
        "evaluation.rolling_origin.step=168",
    ],
)
cfg_wcal = load_config(
    config_path=REPO_ROOT / "conf",
    overrides=[
        "features=weather_calendar",
        "evaluation.rolling_origin.min_train_periods=720",
        "evaluation.rolling_origin.step=168",
    ],
)
assert cfg_wonly.features.weather_only is not None, "features.weather_only not resolved"
assert cfg_wcal.features.weather_calendar is not None, "features.weather_calendar not resolved"

weather_only_path = (
    cfg_wonly.features.weather_only.cache_dir / cfg_wonly.features.weather_only.cache_filename
)
weather_calendar_path = (
    cfg_wcal.features.weather_calendar.cache_dir
    / cfg_wcal.features.weather_calendar.cache_filename
)
features_wonly = assembler.load(weather_only_path).set_index("timestamp_utc")
features_wcal = assembler.load_calendar(weather_calendar_path).set_index("timestamp_utc")

weather_column_names = tuple(name for name, _ in assembler.WEATHER_VARIABLE_COLUMNS)
calendar_column_names = tuple(name for name, _ in CALENDAR_VARIABLE_COLUMNS)

print(
    f"weather_only   : rows={len(features_wonly):,}  feature columns={len(weather_column_names)}"
)
print(
    f"weather_calendar: rows={len(features_wcal):,}  feature columns={len(weather_column_names) + len(calendar_column_names)}"
)
print(f"Window: {features_wcal.index[0]} -> {features_wcal.index[-1]}")
print(
    f"Calendar column groups: hour-of-day=23, day-of-week=6, month=11, holiday=4 (total {len(calendar_column_names)})"
)

## `weather_only` run — the Stage 4 baseline

The Stage 4 notebook already established that a weather-only OLS misses the weekly rhythm. Rerun it here as the reference point. `harness.evaluate` calls `model.fit` / `model.predict` once per rolling-origin fold and returns a long-form per-fold DataFrame; the four metric columns are fractions (WAPE / MAPE) or MW (MAE / RMSE) per plan DESIGN §5.3.

In [ ]:
metric_names = (
    cfg_wonly.evaluation.metrics.names
    if cfg_wonly.evaluation.metrics is not None
    else tuple(METRIC_REGISTRY)
)
metric_fns = [METRIC_REGISTRY[name] for name in metric_names]
split_cfg = cfg_wonly.evaluation.rolling_origin

linear_wonly = LinearModel(LinearConfig(feature_columns=weather_column_names))
wonly_per_fold = evaluate(
    linear_wonly,
    features_wonly,
    split_cfg,
    metric_fns,
    feature_columns=weather_column_names,
)
print(f"weather_only mean per metric: {wonly_per_fold[list(metric_names)].mean().to_dict()}")
print(linear_wonly.results.summary())
wonly_per_fold.head()

## `weather_calendar` run — the teaching-moment table

The `results.summary()` block below is the Stage 5 demo payoff. Three things to point at:

1. **The hour-of-day dummies** climb from `hour_of_day_05` (the pre-dawn trough) to a peak around `hour_of_day_17`–`hour_of_day_18` (the GB evening peak). The coefficient magnitudes read the GB load profile off the regression.
2. **The day-of-week dummies** dip on Saturday and Sunday relative to Monday (the `day_of_week_0` reference). Tuesday / Wednesday / Thursday sit close to Monday — the midweek plateau.
3. **`is_bank_holiday_ew`** carries a large negative coefficient (E&W accounts for ~85 % of GB demand). `is_bank_holiday_sco` is a weaker contribution — Scotland's ~10 % share means its holiday effect moves the aggregate less.

`is_weekend` is **deliberately not emitted** from the calendar derivation (external research §R5 — it is perfectly collinear with the sum of the Saturday and Sunday day-of-week dummies). The invariant is pinned by a module-level assertion in `bristol_ml.features.calendar`.

In [ ]:
wcal_feature_cols = weather_column_names + calendar_column_names
linear_wcal = LinearModel(LinearConfig(feature_columns=wcal_feature_cols))
wcal_per_fold = evaluate(
    linear_wcal,
    features_wcal,
    split_cfg,
    metric_fns,
    feature_columns=wcal_feature_cols,
)
print(f"weather_calendar mean per metric: {wcal_per_fold[list(metric_names)].mean().to_dict()}")
print(linear_wcal.results.summary())
wcal_per_fold.head()

## Side-by-side — what domain knowledge bought us

Mean metric across rolling-origin folds. MAPE and WAPE are fractions (0.05 = 5 %); MAE and RMSE are in MW.

In [ ]:
side_by_side = pd.DataFrame(
    {
        "weather_only": wonly_per_fold[list(metric_names)].mean(),
        "weather_calendar": wcal_per_fold[list(metric_names)].mean(),
    }
).T
side_by_side["mape_improvement_pp"] = (
    side_by_side.loc["weather_only", "mape"] - side_by_side["mape"]
) * 100.0
side_by_side

## Three-way benchmark — calendar vs NESO

If the NESO day-ahead forecast archive is cached (`python -m bristol_ml.ingestion.neso_forecast --cache auto`), score both linear models against the published NESO forecast on the holdout window. Per plan D4, the half-hourly NESO series is aggregated to hourly via `mean`. Every row is evaluated on the intersection of the fold test periods with the forecast archive's coverage.

In [ ]:
forecast_cache = (
    cfg_wcal.ingestion.neso_forecast.cache_dir
    / cfg_wcal.ingestion.neso_forecast.cache_filename
    if cfg_wcal.ingestion.neso_forecast is not None
    else None
)
if forecast_cache is not None and forecast_cache.exists():
    neso_df = neso_forecast_mod.load(forecast_cache)
    benchmark_table_wonly = compare_on_holdout(
        {"linear_weather_only": LinearModel(LinearConfig(feature_columns=weather_column_names))},
        features_wonly,
        neso_df,
        split_cfg,
        metric_fns,
        aggregation=(
            cfg_wonly.evaluation.benchmark.aggregation
            if cfg_wonly.evaluation.benchmark is not None
            else "mean"
        ),
        feature_columns=weather_column_names,
    )
    benchmark_table_wcal = compare_on_holdout(
        {"linear_weather_calendar": LinearModel(LinearConfig(feature_columns=wcal_feature_cols))},
        features_wcal,
        neso_df,
        split_cfg,
        metric_fns,
        aggregation=(
            cfg_wcal.evaluation.benchmark.aggregation
            if cfg_wcal.evaluation.benchmark is not None
            else "mean"
        ),
        feature_columns=wcal_feature_cols,
    )
    benchmark_table = pd.concat(
        [
            benchmark_table_wonly.loc[["linear_weather_only"]],
            benchmark_table_wcal.loc[["linear_weather_calendar"]],
            benchmark_table_wcal.loc[["neso"]],
        ]
    )
    print(benchmark_table.to_string(float_format=lambda v: f"{v:.3f}"))
else:
    print(
        "NESO forecast cache not populated -- run "
        "`python -m bristol_ml.ingestion.neso_forecast --cache auto` to fetch the "
        "archive, then re-run this cell."
    )
    benchmark_table = None

## Residual ripple — before and after calendar features

Acceptance criterion 5: visualise how calendar features flatten the weekly oscillation. Both linear models are fit on the full feature table (a single in-sample fit, not a rolling-origin fold) and their residuals are plotted over the last test week in `Europe/London` local time. The `weather_only` residuals carry a visible ~7-day ripple — the effect of Saturday / Sunday / bank-holiday days that the regressor cannot see. The `weather_calendar` residuals largely flatten that rhythm.

In [ ]:
linear_wonly_full = LinearModel(LinearConfig(feature_columns=weather_column_names))
linear_wonly_full.fit(features_wonly, features_wonly["nd_mw"])
linear_wcal_full = LinearModel(LinearConfig(feature_columns=wcal_feature_cols))
linear_wcal_full.fit(features_wcal, features_wcal["nd_mw"])

window_end = features_wcal.index[-1]
window_start = window_end - pd.Timedelta(hours=7 * 24 - 1)
window_wonly = features_wonly.loc[window_start:window_end]
window_wcal = features_wcal.loc[window_start:window_end]

residual_wonly = window_wonly["nd_mw"] - linear_wonly_full.predict(window_wonly)
residual_wcal = window_wcal["nd_mw"] - linear_wcal_full.predict(window_wcal)
display_ts = window_wcal.index.tz_convert("Europe/London")

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0.0, color="grey", linewidth=0.8)
ax.plot(
    display_ts,
    residual_wonly,
    color="C1",
    linewidth=1.0,
    label=f"weather_only (std={residual_wonly.std():,.0f} MW)",
)
ax.plot(
    display_ts,
    residual_wcal,
    color="C2",
    linewidth=1.0,
    label=f"weather_calendar (std={residual_wcal.std():,.0f} MW)",
)
ax.set_xlabel("Local time (Europe/London)")
ax.set_ylabel("Residual (MW)")
ax.set_title("Residuals on the final test week -- weekly ripple before and after calendar features")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## What calendar features did and did not buy us

**Bought us** — the weekly ripple, the daily peak / trough shape, the bank-holiday gouges, and the annual seasonal envelope, all encoded as one-hot dummies that a single OLS regression consumes without modification. The Stage 4 `LinearModel` class, `harness.evaluate`, `compare_on_holdout`, the `Model` protocol, and every piece of `SplitterConfig` plumbing stay fixed — only the config resolved the other feature set.

**Did not buy us** — nothing about feature *interactions* (hot Saturdays vs cold Mondays look the same to a main-effects-only OLS) and nothing about *lags* of the target (yesterday's demand is a strong predictor of today's; not used here). Stage 6 lifts the linearity constraint with tree-based models so the interactions come through automatically; Stage 7 adds explicit lag features. One-off events — REMIT notifications, royal funerals, the 2022 Commonwealth Games — remain out of scope for this stage.

### Things to try in a live demo

- Run `python -m bristol_ml.train features=weather_calendar` from the terminal and compare the per-fold table to the Stage 4 `python -m bristol_ml.train` output — the delta is the calendar block's entire contribution, isolated from any other change.
- Scroll to the Easter Monday bank holiday in the residual plot (last week of the year the notebook happens to land on a non-bank-holiday week; shift the window to April 2023 to see the effect).
- Comment out the `is_bank_holiday_ew` column from the feature list and watch the regression MAPE worsen visibly — a direct live-demo illustration of what that one column is earning.